<a href="https://colab.research.google.com/github/mattsmiths/biol470/blob/main/week8/BIOL470_spikeProc_SpikeAnalysis_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [130]:
#@title Click to copy the class code repository
!git clone https://github.com/mattsmiths/biol470.git
finalPlot = {}

fatal: destination path 'biol470' already exists and is not an empty directory.


In [ ]:
import matplotlib.pyplot as plt
import librosa
import librosa.display
import numpy as np
import os
import pickle

# New Analysis


In [ ]:
import scipy.signal as sg
from scipy.signal import find_peaks

In [246]:

select_goup = 'odor_during' # @param ["control_during", "control_after","control_pre","odor_during","odor_after","odor_pre"]

if select_goup == 'control_during':
  videoPath = '/content/biol470/week8/electrode_data/2026/air_present_2026'
elif select_goup == 'control_after':
  videoPath = '/content/biol470/week8/electrode_data/2026/air_after_2026'
elif select_goup == 'odor_during':
  videoPath = '/content/biol470/week8/electrode_data/2026/odor_present_2026'
elif select_goup == 'odor_after':
  videoPath = '/content/biol470/week8/electrode_data/2026/odor_after_2026'
elif select_goup == 'control_pre':
  videoPath = '/content/biol470/week8/electrode_data/2026/pre_air_2026'
elif select_goup == 'odor_pre':
  videoPath = '/content/biol470/week8/electrode_data/2026/pre_odor_2026'

in1 = open(videoPath,'rb')
pulse_data = pickle.load(in1)
in1.close()

if os.path.isdir('/content/%s/'%(videoPath.split('/')[-1])) == False:
  os.mkdir('/content/%s/'%(videoPath.split('/')[-1]))

In [247]:
# add a highpass filter for noise in data
b, a = sg.butter(4, 100. / (7500 / 2.), 'highpass')
if 'pre_' in videoPath:
  x_fil = sg.filtfilt(b, a, pulse_data[4])
else:
  x_fil = sg.filtfilt(b, a, pulse_data[0][0])

In [248]:
data_full = []

if 'pre_' in videoPath:
  for ele in np.arange(4,len(pulse_data),4):
    data_full+=list(pulse_data[ele])
  data_full = np.array(data_full)[10000:]
else:
  for ee in  pulse_data[0]:
      data_full+=list(ee)

dtest = data_full[:35000] - np.nanmean(data_full[:35000])
x_fil_II = sg.filtfilt(b, a, dtest)
thresh = np.nanmean(x_fil_II)+(np.nanstd(x_fil_II)*2)
peaks, _ = find_peaks(x_fil_II*-1,height=thresh)

In [ ]:
plt.figure(figsize=(7,4))
plt.plot(x_fil_II,linewidth=0.5,color=(0.25,0.25,0.25,0.5))
plt.title('Filtered Data')
plt.xlabel('Time (seconds)')
plt.ylabel('Voltage (mV)')
#xticks = [0,10000,20000,30000,40000,50000,60000,70000,80000,90000]
xticks = np.arange(0,len(x_fil_II),7500)
xlabels = np.arange(0,len(xticks))
l = plt.xticks(xticks,xlabels)

a1 = plt.axis()
plt.plot([a1[0],a1[1]],[np.nanmean(x_fil_II)-(np.nanstd(x_fil_II)*2),np.nanmean(x_fil_II)-(np.nanstd(x_fil_II)*2)],linestyle='--',color=(0.5,0,0,0.5))
plt.plot()
for ele in peaks:
    plt.scatter(ele,x_fil_II[ele],marker='o',s=50,color=(0.65,0.2,0.2,0.5))

plt.savefig('/content/%s/%s_spikePeaks.png'%(videoPath.split('/')[-1],videoPath.split('/')[-1]),dpi=100)

In [ ]:
plt.figure()
window1 = 200
tt = []
for ele in peaks:
    if ele - window1 <=0:continue
    if ele + window1 >=len(x_fil_II):continue
    plt.plot(np.arange(window1*2),x_fil_II[ele-window1:ele+window1],linewidth=0.5,color=(0.3,0.3,0.3,0.3))
    tt.append(dtest[ele-window1:ele+window1])
airT = tt.copy()
a1 = plt.axis()
#plt.plot([window1,window1],[a1[2],a1[3]],linewidth=3,linestyle='--',color=(0.65,0.2,0.2,0.75),label='alignment')
plt.legend()

plt.plot(np.arange(window1*2),np.nanmean(tt,0),linewidth=3,color=(0.1,0.1,0.1,0.9))

tt = ((window1/(13*50))*2)+1
tt = np.arange(0,window1*2,37.5)
oo = plt.xticks(tt,np.arange(0,0.5*len(tt),0.5))
plt.xlabel('milliseconds')
plt.ylabel('voltage')
plt.plot([a1[0],a1[1]],[0,0],linewidth=1,linestyle='--',color=(0.2,0.2,0.5,0.5),label='alignment')
plt.title(videoPath.split('/')[-1])
plt.savefig('spikes_aligned.png',dpi=300)
#plt.savefig('/content/%s/%s_spikePeaks.png'%(videoPath.split('/')[-1],videoPath.split('/')[-1]),dpi=300)

In [ ]:
#@title Processing full dataset
bgg = []
nS = []
nA = []

if 'pre_' in videoPath:

  for epp in range(22500,len(data_full),22500):

    dtest = data_full[epp:epp+22500] - np.nanmean(data_full[epp:epp+22500])


    x_fil_II = sg.filtfilt(b, a, dtest)
    thresh = np.nanmean(x_fil_II)+(np.nanstd(x_fil_II)*2)
    peaks, _ = find_peaks(x_fil_II*-1,height=thresh)

    plt.figure()

    window1 = 200
    tt = []
    nS.append(len(peaks))

    for ele in peaks:
        if ele - window1 <=0:continue
        if ele + window1 >=len(x_fil_II):continue
        plt.plot(np.arange(window1*2),x_fil_II[ele-window1:ele+window1],linewidth=0.5,color=(0.3,0.3,0.3,0.3))
        tt.append(dtest[ele-window1:ele+window1])
        bgg.append(dtest[ele-window1:ele+window1])
        nA.append(np.nanmax(dtest[ele-window1:ele+window1]))
    airT = tt.copy()
    a1 = plt.axis()
    #plt.plot([window1,window1],[a1[2],a1[3]],linewidth=3,linestyle='--',color=(0.65,0.2,0.2,0.75),label='alignment')
    #plt.legend()

    plt.plot(np.arange(window1*2),np.nanmean(tt,0),linewidth=3,color=(0.1,0.1,0.7,0.9))

    tt = ((window1/(13*50))*2)+1
    tt = np.arange(0,window1*2,37.5)
    oo = plt.xticks(tt,np.arange(0,0.5*len(tt),0.5))
    plt.xlabel('milliseconds')
    plt.ylabel('voltage')
    plt.plot([a1[0],a1[1]],[0,0],linewidth=1,linestyle='--',color=(0.2,0.2,0.5,0.5),label='alignment')
    plt.title(videoPath.split('/')[-1])

else:
  for ppu in pulse_data:

      data_full = []
      for ee in ppu[:2]:
          data_full+=list(ee)

      dtest = data_full[:35000] - np.nanmean(data_full[:35000])
      x_fil_II = sg.filtfilt(b, a, dtest)
      thresh = np.nanmean(x_fil_II)+(np.nanstd(x_fil_II)*2)
      peaks, _ = find_peaks(x_fil_II*-1,height=thresh)

      plt.figure()

      window1 = 200
      tt = []
      nS.append(len(peaks))

      for ele in peaks:
          if ele - window1 <=0:continue
          if ele + window1 >=len(x_fil_II):continue
          plt.plot(np.arange(window1*2),x_fil_II[ele-window1:ele+window1],linewidth=0.5,color=(0.3,0.3,0.3,0.3))
          tt.append(dtest[ele-window1:ele+window1])
          bgg.append(dtest[ele-window1:ele+window1])
          nA.append(np.nanmax(dtest[ele-window1:ele+window1]))
      airT = tt.copy()
      a1 = plt.axis()
      #plt.plot([window1,window1],[a1[2],a1[3]],linewidth=3,linestyle='--',color=(0.65,0.2,0.2,0.75),label='alignment')
      #plt.legend()

      plt.plot(np.arange(window1*2),np.nanmean(tt,0),linewidth=3,color=(0.1,0.1,0.7,0.9))

      tt = ((window1/(13*50))*2)+1
      tt = np.arange(0,window1*2,37.5)
      oo = plt.xticks(tt,np.arange(0,0.5*len(tt),0.5))
      plt.xlabel('milliseconds')
      plt.ylabel('voltage')
      plt.plot([a1[0],a1[1]],[0,0],linewidth=1,linestyle='--',color=(0.2,0.2,0.5,0.5),label='alignment')
      plt.title(videoPath.split('/')[-1])

In [ ]:
for ele in bgg:
  plt.plot(np.arange(window1*2),ele,linewidth=0.5,color=(0.3,0.3,0.3,0.3))

plt.plot(np.arange(window1*2),np.nanmean(bgg,0),linewidth=3,color=(0.1,0.1,0.7,0.9))
tt = ((window1/(13*50))*2)+1
tt = np.arange(0,window1*2,37.5)
oo = plt.xticks(tt,np.arange(0,0.5*len(tt),0.5))
plt.xlabel('milliseconds')
plt.ylabel('voltage')
plt.plot([a1[0],a1[1]],[0,0],linewidth=1,linestyle='--',color=(0.2,0.2,0.5,0.5),label='alignment')
plt.title(videoPath.split('/')[-1])
plt.ylim([-1750,1750])
plt.savefig('/content/%s/%s_average.png'%(videoPath.split('/')[-1],videoPath.split('/')[-1]),dpi=100)

In [253]:
sem1 = np.nanmean(nS)/np.sqrt(len(nS))
finalPlot[videoPath.split('/')[-1]] = [np.nanmean(nS),sem1]

In [ ]:
plt.figure(figsize=(5,5),dpi=300)
plt.ylabel('Number of detected "spikes" in 5 seconds')


sem1 = np.nanmean(nS)/np.sqrt(len(nS))
plt.plot([0,0],[np.nanmean(nS)-sem1,np.nanmean(nS)+sem1],color=(0.35,0.35,0.35,0.75))


plt.scatter([0],[np.nanmean(nS)],color=(0.35,0.35,0.35),s=150)

plt.xticks([0],['%s'%videoPath.split('/')[-1]])
plt.axis([-0.25,1.25,0,200])
plt.savefig('/content/%s/SpikeNumber%s.png'%(videoPath.split('/')[-1],videoPath.split('/')[-1]),dpi=300)

In [255]:
import pandas as pd

tout = {}
tout['data'] = x_fil_II
tout2 = pd.DataFrame.from_dict(tout)
tout2.to_csv('/content/%s/csv_data_%s.csv'%(videoPath.split('/')[-1],videoPath.split('/')[-1]))

In [ ]:
# @title
# Making comparitive plot

plt.figure()
oc = 0
cb = 0

xtb = []
ytb = []

kkeys = finalPlot.keys()
p1 = []
p2 = []
p3 = []
for ele in kkeys:
  if 'pre_' in ele:
    p1.append(ele)
  elif 'present' in ele:
    p2.append(ele)
  else:
    p3.append(ele)
lList = p1+p2+p3

for ind,ele in enumerate(lList):

  if 'odor' in ele:
    cc = (0.65,0.2,0.2,0.75)
    ii = 1+oc
    oc+=0.15
  else:
    cc = (0.35,0.35,0.35,0.75)
    ii = 0+cb
    cb+=0.15

  plt.plot([ii,ii],[finalPlot[ele][0]+finalPlot[ele][1],finalPlot[ele][0]-finalPlot[ele][1]],color=cc)
  plt.scatter([ii],[finalPlot[ele][0]],color=cc,s=150)

  xtb.append(ii)
  ytb.append(ele)
plt.xticks(xtb,ytb,rotation=60)
plt.xlim([-0.25,1.5])
plt.ylabel('Number of detected "spikes" in 2 seconds')
plt.tight_layout()
plt.savefig('/content/comparison.png',dpi=300)